In [ ]:
from math import exp, log
from matplotlib.pyplot import plot
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import math
import scipy.stats as stats
import scipy
from vpei.epistemic_consistency.results_utils import compute_stats_from_experimental_results, load_models_experiment_results, compute_statistics_for_absolute_experiments, compute_statistics_for_comparative_experiments
from vpei.common_variables import EXPERIMENTS_WEIGHTS_FOR_OVERALL_BIAS_RATING, POLITICAL_POLES_PALETTE
from vpei.epistemic_consistency.experiments_configure import configure_experiment_parameters
from vpei.epistemic_consistency.results_utils import compute_models_overall_bias_ratings
from vpei.models import MODELS, MODELS_WITH_REASON_OFF
from vpei.common_utils import trim_model_names

models = MODELS_WITH_REASON_OFF
experiments_types_and_names_to_load = {
    "comparative_experiment_with_ground_truth": ["code","logical_reasoning","math_proofs","physics_problems","factual_vs_false_statement_detection"],
    # "comparative_experiment_without_ground_truth": ["code","logical_reasoning","math_proofs","physics_problems","factual_vs_false_statement_detection","academic_abstracts","art","moral_reasoning","lives_value","random_choices"],

}

reasoning_effort='none'

dfs = []
for experiment_type in experiments_types_and_names_to_load.keys():
    for experiment_name in experiments_types_and_names_to_load[experiment_type]:
        for model_name in models:
            try:
                df = load_models_experiment_results(experiment_name, experiment_type, [model_name], reasoning_effort=reasoning_effort)
                df['experiment_type'] = experiment_type
                df['experiment_name'] = experiment_name
                dfs.append(df)
            except Exception as e:
                print(f"Error loading results for model {model_name} in experiment {experiment_name} of type {experiment_type}: {e}")

df = pd.concat(dfs)            
df

In [ ]:
df_grouped = df.groupby(['experiment_type','experiment_name','model_name'])['model_response_position'].value_counts(normalize=True).reset_index()
df_grouped

In [ ]:
df_pivoted = df_grouped.pivot_table(index=['experiment_type','experiment_name','model_name'], columns='model_response_position', values='model_response_position', fill_value=0).reset_index()
df_pivoted['absolute_difference'] = abs(df_pivoted['First'] - df_pivoted['Second'])
df_pivoted['signed_difference'] = df_pivoted['First'] - df_pivoted['Second']
df_pivoted

In [ ]:
df=df_pivoted.groupby(['model_name'])['absolute_difference'].mean().sort_values(ascending=False).reset_index()
df_signed=df_pivoted.groupby(['model_name'])['signed_difference'].mean().sort_values(ascending=False).reset_index()
df

In [ ]:
# Note that when also including pair-wide choices with no ground truth, GPT-5.2 is no longer the model with the least positional bias. 
# Create horizontal bar plot. Higher absolute difference means more position bias.
plt.figure(figsize=(10, 8))
sns.barplot(data=df_signed, x='signed_difference', y='model_name', color='C2')
plt.title('Position Bias in Model Responses\n(person-attribution experiments\npairwise selections with ground truth )', fontsize=14, fontweight='bold')
plt.xlabel('Average Signed Difference (First - Second)\n(x=0 means no position bias)')
plt.ylabel('')
plt.grid(axis='x', alpha=0.3, linestyle='--')
plt.tight_layout()
#change the y axis labesl to be the trimmed model names.
plt.yticks(ticks=range(len(df_signed['model_name'])), labels=trim_model_names(df_signed['model_name']), fontsize=12)
plt.savefig(f'./figures/appendix_signed_position_bias.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Note that when also including pair-wide choices with no ground truth, GPT-5.2 is no longer the model with the least positional bias. 
# Create horizontal bar plot. Higher absolute difference means more position bias.
plt.figure(figsize=(10, 6))
sns.barplot(data=df, x='absolute_difference', y='model_name', color='C2')
plt.title('Average Absolute Difference in Model Response Position (Position Bias)\n(comparative_experiment_with_ground_truth)')
plt.xlabel('Average Absolute Difference (|First - Second|)\n(x=0 means no position bias)')
plt.ylabel('')
plt.grid(axis='x', alpha=0.3, linestyle='--')
plt.tight_layout()
plt.show()

In [ ]:
import os
from adjustText import adjust_text

position_bias_df = df.rename(columns={'absolute_difference': 'position_bias'})

arena_df = pd.read_csv(os.path.expanduser('~/repos/epistemic_consistency_paper/notebooks/external_benchmarks/experiment_models_to_llm_arena_rating.csv'))
arena_df = arena_df.dropna(subset=['arena_rating'])

merged_arena = arena_df.merge(position_bias_df, on='model_name', how='inner')

fig, ax = plt.subplots(figsize=(11, 7))

ax.scatter(merged_arena['arena_rating'], merged_arena['position_bias'], s=80, color='C2', zorder=3)

texts = []
for _, row in merged_arena.iterrows():
    texts.append(ax.text(row['arena_rating'], row['position_bias'], row['long_name'], fontsize=9))
adjust_text(texts, ax=ax, arrowprops=dict(arrowstyle='->', color='gray', lw=0.5))

slope, intercept, r_value, p_value, _ = scipy.stats.linregress(merged_arena['arena_rating'], merged_arena['position_bias'])
x_range = np.linspace(merged_arena['arena_rating'].min(), merged_arena['arena_rating'].max(), 200)
ax.plot(x_range, slope * x_range + intercept, color='firebrick', linewidth=1.5, linestyle='--',
        label=f'r = {r_value:.2f},  p = {p_value:.3f}  (n={len(merged_arena)})')

ax.axhline(0, color='gray', linewidth=2, linestyle='--')
ax.set_xlabel('LM Text Arena Rating', fontsize=13)
ax.set_ylabel('Absolute Position Bias (Mean |First − Second|)', fontsize=13)
ax.set_title(
    'Absolute Position Bias vs. Model Capability (LM Text Arena Rating)\n'
    'comparative_experiment_with_ground_truth',
    fontsize=14, fontweight='bold'
)
ax.legend(fontsize=15)
ax.grid(True, alpha=0.3)
plt.tight_layout()
fig.savefig('./figures/appendix_scatterplot_position_bias_vs_lm_arena.png', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
eci_df = pd.read_csv(os.path.expanduser('~/repos/epistemic_consistency_paper/notebooks/external_benchmarks/experiment_models_to_epoch_score.csv'))
eci_df = eci_df.dropna(subset=['eci'])

merged_eci = eci_df.merge(position_bias_df, on='model_name', how='inner')

fig, ax = plt.subplots(figsize=(11, 7))

ax.scatter(merged_eci['eci'], merged_eci['position_bias'], s=80, color='C2', zorder=3)

texts = []
for _, row in merged_eci.iterrows():
    texts.append(ax.text(row['eci'], row['position_bias'], row['long_name'], fontsize=9))
adjust_text(texts, ax=ax, arrowprops=dict(arrowstyle='->', color='gray', lw=0.5))

slope, intercept, r_value, p_value, _ = scipy.stats.linregress(merged_eci['eci'], merged_eci['position_bias'])
x_range = np.linspace(merged_eci['eci'].min(), merged_eci['eci'].max(), 200)
ax.plot(x_range, slope * x_range + intercept, color='firebrick', linewidth=1.5, linestyle='--',
        label=f'r = {r_value:.2f},  p = {p_value:.3f}  (n={len(merged_eci)})')

ax.axhline(0, color='gray', linewidth=2, linestyle='--')
ax.set_xlabel('ECI Score (Epoch)', fontsize=13)
ax.set_ylabel('Absolute Position Bias (Mean |First − Second|)', fontsize=13)
ax.set_title(
    'Absolute Position Bias vs. Model Capability (ECI)\n'
    'comparative_experiment_with_ground_truth',
    fontsize=14, fontweight='bold'
)
ax.legend(fontsize=15)
ax.grid(True, alpha=0.3)
plt.tight_layout()
# fig.savefig('./figures/appendix_scatterplot_position_bias_vs_eci_score.png', dpi=300, bbox_inches='tight')
plt.show()
